In [1]:
import math
import time
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --------------------
# symdisc
# --------------------
from symdisc import generate_euclidean_killing_fields_with_names
from symdisc.enforcement.regularization.penalties import forward_with_equivariance_penalty
from symdisc.enforcement.regularization.utilities import as_field_lastdim, make_pairer
from symdisc.enforcement.regularization.diagonal import sum_fields
from symdisc.enforcement.regularization import schedules as S

# --------------------
# e3nn (optional)
# --------------------
try:
    from e3nn.o3 import Irreps, Linear, FullyConnectedTensorProduct
    from e3nn.nn import NormActivation
    from e3nn.nn import Gate
    _HAS_E3NN = True
except Exception as _e:
    _HAS_E3NN = False
    _E3NN_IMPORT_ERR = _e

In [2]:
# =============================================================================
# Reproducibility
# =============================================================================

def set_seed(seed: int = 1337):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
# =============================================================================
# Mandel helpers: [M11, s2*M12, s2*M13, M22, s2*M23, M33]
# =============================================================================

_SQRT2 = math.sqrt(2.0)


def mandel_to_mat(m: torch.Tensor) -> torch.Tensor:
    """m (...,6) -> (...,3,3) symmetric."""
    M11 = m[..., 0]
    M12 = m[..., 1] / _SQRT2
    M13 = m[..., 2] / _SQRT2
    M22 = m[..., 3]
    M23 = m[..., 4] / _SQRT2
    M33 = m[..., 5]
    out = torch.zeros((*m.shape[:-1], 3, 3), device=m.device, dtype=m.dtype)
    out[..., 0, 0] = M11
    out[..., 0, 1] = M12
    out[..., 0, 2] = M13
    out[..., 1, 0] = M12
    out[..., 1, 1] = M22
    out[..., 1, 2] = M23
    out[..., 2, 0] = M13
    out[..., 2, 1] = M23
    out[..., 2, 2] = M33
    return out


def mat_to_mandel(M: torch.Tensor) -> torch.Tensor:
    """M (...,3,3) -> (...,6) in the user Mandel ordering."""
    return torch.stack([
        M[..., 0, 0],
        _SQRT2 * M[..., 0, 1],
        _SQRT2 * M[..., 0, 2],
        M[..., 1, 1],
        _SQRT2 * M[..., 1, 2],
        M[..., 2, 2],
    ], dim=-1)

In [4]:
# =============================================================================
# Data
# =============================================================================

@dataclass
class DataSplit:
    X_train: torch.Tensor
    Y_train: torch.Tensor
    X_test: torch.Tensor
    Y_test: torch.Tensor


class TabularTensorDataset(Dataset):
    def __init__(self, X: torch.Tensor, Y: torch.Tensor):
        assert X.ndim == 2
        assert Y.ndim == 2 and Y.shape[1] == 6
        self.X = X
        self.Y = Y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


def load_squareduct_pointwise(
    data_csv: str | Path,
    coord_csv: str | Path,
    *,
    grad_cols: Optional[List[str]] = None,
    out_cols: Optional[List[str]] = None,
    dtype: torch.dtype = torch.float32,
) -> DataSplit:
    """Load gradients and targets from data_csv and y from coord_csv; split by sign(y).

    Assumes row alignment between the two CSV files.

    Defaults:
      grad_cols = [REF_gradU_11..REF_gradU_33] (9 columns)
      out_cols  = [w1..w6] (6 columns in your Mandel ordering)
    """
    data_csv = Path(data_csv)
    coord_csv = Path(coord_csv)

    df = pd.read_csv(data_csv)
    coords = pd.read_csv(coord_csv)

    if 'y' not in coords.columns:
        raise ValueError(f"coord_csv must contain a 'y' column; got {list(coords.columns)}")
    if len(df) != len(coords):
        raise ValueError(
            f"Row mismatch: data_csv has {len(df)} rows but coord_csv has {len(coords)} rows. "
            "Provide aligned files (same row ordering)."
        )

    if out_cols is None:
        cand = [f"w{i}" for i in range(1, 7)]
        if all(c in df.columns for c in cand):
            out_cols = cand
        else:
            raise ValueError(
                f"Could not infer output columns w1..w6. Present columns include: {list(df.columns)[:30]} ..."
            )

    if grad_cols is None:
        # common naming in your dataset
        cand = [
            'REF_gradU_11','REF_gradU_12','REF_gradU_13',
            'REF_gradU_21','REF_gradU_22','REF_gradU_23',
            'REF_gradU_31','REF_gradU_32','REF_gradU_33'
        ]
        if all(c in df.columns for c in cand):
            grad_cols = cand
        else:
            # fallback: try without REF_ prefix
            cand2 = ['u11','u12','u13','u21','u22','u23','u31','u32','u33']
            if all(c in df.columns for c in cand2):
                grad_cols = cand2
            else:
                raise ValueError(
                    "Could not infer 9 gradient columns. Provide grad_cols explicitly."
                )

    X = torch.tensor(df[grad_cols].to_numpy(), dtype=dtype)
    Y = torch.tensor(df[out_cols].to_numpy(), dtype=dtype)
    ycoord = coords['y'].to_numpy().astype(np.float32)

    train_mask = ycoord >= 0.0
    test_mask = ~train_mask

    X_train = X[train_mask]
    Y_train = Y[train_mask]
    X_test  = X[test_mask]
    Y_test  = Y[test_mask]

    return DataSplit(X_train=X_train, Y_train=Y_train, X_test=X_test, Y_test=Y_test)

In [5]:
# =============================================================================
# Vector-field generators (SO(2) about streamwise axis)
# =============================================================================

def build_single_node_generator_X3() -> Callable:
    """Generator acting on the 9D flattened gradU in order:
    [u11,u12,u13,u21,u22,u23,u31,u32,u33]

    Uses symdisc Euclidean Killing fields and pairers, exactly as in your sandbox code.
    """
    d = 9
    pair_node = make_pairer(["u11","u12","u13","u21","u22","u23","u31","u32","u33"])

    fields, names = generate_euclidean_killing_fields_with_names(
        d=d, include_translations=False, include_rotations=True, backend="torch"
    )
    name_to_field = {n: f for f, n in zip(fields, names)}

    def R(i, j):
        key = f"R_{i}_{j}" if i < j else f"R_{j}_{i}"
        return as_field_lastdim(name_to_field[key], d=d)

    return sum_fields(
        R(*pair_node("u22","u23")),
        R(*pair_node("u12","u13")),
        R(*pair_node("u23","u33")),
        R(*pair_node("u32","u33")),
        R(*pair_node("u22","u32")),
        R(*pair_node("u21","u31")),
    )


def build_single_output_generator_Y3() -> Callable:
    """Generator acting on Mandel-6 outputs in ordering:
    [M11, s2*M12, s2*M13, M22, s2*M23, M33]

    Uses your existing pairing logic.
    """
    d = 6
    pair_out = make_pairer(["w1","w2","w3","w4","w5","w6"])

    fields, names = generate_euclidean_killing_fields_with_names(
        d=d, include_translations=False, include_rotations=True, backend="torch"
    )
    name_to_field = {n: f for f, n in zip(fields, names)}

    def R(i, j):
        key = f"R_{i}_{j}" if i < j else f"R_{j}_{i}"
        return as_field_lastdim(name_to_field[key], d=d)

    # matches your previous construction for Mandel ordering
    s2 = math.sqrt(2.0)
    return sum_fields(
        R(*pair_out("w4","w5")),
        R(*pair_out("w5","w6")),
        R(*pair_out("w2","w3")),
        weights=[s2, s2, 1.0],
    )

In [6]:
# =============================================================================
# Models
# =============================================================================

class BaselineMLP(nn.Module):
    """Shared architecture for baseline and regularized baseline."""
    def __init__(self, hidden: int = 256, depth: int = 4, dropout: float = 0.0, layernorm: bool = True):
        super().__init__()
        layers: List[nn.Module] = []
        in_dim = 9
        for i in range(depth):
            layers.append(nn.Linear(in_dim, hidden))
            if layernorm:
                layers.append(nn.LayerNorm(hidden))
            layers.append(nn.SiLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = hidden
        layers.append(nn.Linear(in_dim, 6))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [7]:
# --------------------
# e3nn pointwise model
# --------------------

def _decompose_grad_u(x9: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U = x9.view(*x9.shape[:-1], 3, 3)
    S = 0.5 * (U + U.transpose(-1, -2))
    W = 0.5 * (U - U.transpose(-1, -2))
    return S, W


def _sym_to_trace_l2(S: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    Sxx, Syy, Szz = S[...,0,0], S[...,1,1], S[...,2,2]
    Sxy, Sxz, Syz = S[...,0,1], S[...,0,2], S[...,1,2]
    tr = (Sxx + Syy + Szz).unsqueeze(-1)
    a0  = (2.0*Szz - Sxx - Syy) / math.sqrt(6.0)
    a2c = (Sxx - Syy) / math.sqrt(2.0)
    a2s = math.sqrt(2.0) * Sxy
    a1c = math.sqrt(2.0) * Sxz
    a1s = math.sqrt(2.0) * Syz
    a = torch.stack([a0, a2c, a2s, a1c, a1s], dim=-1)
    return tr, a


def _irreps_to_mandel(trace_0e: torch.Tensor, a_l2: torch.Tensor) -> torch.Tensor:
    a0, a2c, a2s, a1c, a1s = a_l2.unbind(dim=-1)
    dev_xx = -a0 / math.sqrt(6.0) + a2c / math.sqrt(2.0)
    dev_yy = -a0 / math.sqrt(6.0) - a2c / math.sqrt(2.0)
    dev_zz =  2.0 * a0 / math.sqrt(6.0)
    dev_xy =  a2s / math.sqrt(2.0)
    dev_xz =  a1c / math.sqrt(2.0)
    dev_yz =  a1s / math.sqrt(2.0)

    tr = trace_0e.squeeze(-1)
    Sxx = dev_xx + tr / 3.0
    Syy = dev_yy + tr / 3.0
    Szz = dev_zz + tr / 3.0
    Sxy, Sxz, Syz = dev_xy, dev_xz, dev_yz

    return torch.stack([Sxx, _SQRT2*Sxy, _SQRT2*Sxz, Syy, _SQRT2*Syz, Szz], dim=-1)


'''class E3NNPointwise(nn.Module):
    """Pointwise equivariant regression using e3nn irreps.

    - Input: gradU (9) -> typed features (0e + 2e + 1e)
    - Core: equivariant MLP in irreps space (with NormActivation gates)
    - Output: (0e + 2e) -> Mandel(6)

    This is the right mechanism: symmetric rank-2 tensors correspond to (l=0 + l=2).
    """
    def __init__(self, hidden_spec: str = "128x0e + 64x1e + 64x2e + 32x3e", depth: int = 4):
        super().__init__()
        if not _HAS_E3NN:
            raise ImportError(
                "e3nn is required for E3NNPointwise. "
                f"Original import error: {_E3NN_IMPORT_ERR}"
            )

        self.ir_in  = Irreps("0e + 2e + 1e")
        self.ir_hid = Irreps(hidden_spec)
        self.ir_out = Irreps("0e + 2e")

        self.enc = Linear(self.ir_in, self.ir_hid, biases=True)
        blocks: List[nn.Module] = []
        for _ in range(max(depth - 1, 1)):
            #blocks.append(NormActivation(
            #    irreps_in=self.ir_hid,
            #    scalar_nonlinearity=nn.SiLU(),
            #    gate_nonlinearity=nn.Sigmoid(),
            #    normalize=True,
            #))
            blocks.append(NormActivation(
                irreps_in=self.ir_hid,
                scalar_nonlinearity=nn.SiLU(),
                normalize=True,
            ))
            blocks.append(Linear(self.ir_hid, self.ir_hid, biases=True))
        self.core = nn.Sequential(*blocks)
        self.dec = Linear(self.ir_hid, self.ir_out, biases=True)

    @staticmethod
    def featurize(x9: torch.Tensor) -> torch.Tensor:
        S, W = _decompose_grad_u(x9)
        trS, a2 = _sym_to_trace_l2(S)
        # axial vector from W (so(3) dual)
        wx, wy, wz = W[...,1,2], W[...,2,0], W[...,0,1]
        omega = torch.stack([wx, wy, wz], dim=-1)
        return torch.cat([trS, a2, omega], dim=-1)  # 1 + 5 + 3 = 9

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h0 = self.featurize(x)
        h = self.enc(h0)
        h = self.core(h)
        out = self.dec(h)
        tr0 = out[..., 0:1]
        a2  = out[..., 1:6]
        return _irreps_to_mandel(tr0, a2)'''


class E3NNPointwise(nn.Module):
    """
    Fast pointwise equivariant regression using Gate blocks (no tensor products).
    Compatible with e3nn==0.6.0.

    Input: gradU (9) -> typed features (0e + 2e + 1e)
    Hidden: hidden_spec split into scalars (0e) and gated (l>0)
    Output: (0e + 2e) -> Mandel(6) using _irreps_to_mandel
    """

    def __init__(self, hidden_spec: str = "128x0e + 64x1e + 64x2e + 32x3e", depth: int = 4):
        super().__init__()
        if not _HAS_E3NN:
            raise ImportError(
                "e3nn is required for E3NNPointwise. "
                f"Original import error: {_E3NN_IMPORT_ERR}"
            )

        self.ir_in  = Irreps("0e + 2e + 1e")
        self.ir_out = Irreps("0e + 2e")

        hid = Irreps(hidden_spec).simplify()

        # Split hidden irreps into scalar (l=0) and gated (l>0)
        scalars = Irreps([(mul, ir) for mul, ir in hid if ir.l == 0]).simplify()
        gated   = Irreps([(mul, ir) for mul, ir in hid if ir.l > 0]).simplify()

        # Ensure we have some scalar channels (needed by Gate)
        if scalars.dim == 0:
            scalars = Irreps("64x0e")

        # --- CRITICAL FIX ---
        # Gate requires ONE gate scalar per *copy* of each gated irrep:
        # gates.num_irreps must equal gated.num_irreps (multiplicities count)
        n_gates = gated.num_irreps
        if n_gates == 0:
            # ensure we have something to gate
            gated = Irreps("64x2e").simplify()
            n_gates = gated.num_irreps

        gates = Irreps(f"{n_gates}x0e")

        # Activation lists must match the NUMBER OF TERMS (len(Irreps)),
        # not the multiplicity count.
        act_scalars = [F.silu] * len(scalars)          # usually length 1 (e.g., "128x0e")
        act_gates   = [torch.sigmoid] * len(gates)     # usually length 1 (e.g., "160x0e")

        self.gate = Gate(
            irreps_scalars=scalars,
            act_scalars=act_scalars,
            irreps_gates=gates,
            act_gates=act_gates,
            irreps_gated=gated
        )

        self.ir_block_in  = self.gate.irreps_in
        self.ir_block_out = self.gate.irreps_out

        self.enc = Linear(self.ir_in, self.ir_block_in, biases=True)

        # Repeat (Linear -> Gate) blocks
        n_layers = max(int(depth), 1)
        self.layers = nn.ModuleList([
            Linear(self.ir_block_out, self.ir_block_in, biases=True)
            for _ in range(n_layers - 1)
        ])

        self.dec = Linear(self.ir_block_out, self.ir_out, biases=True)

    @staticmethod
    def featurize(x9: torch.Tensor) -> torch.Tensor:
        S, W = _decompose_grad_u(x9)
        trS, a2 = _sym_to_trace_l2(S)
        wx, wy, wz = W[..., 1, 2], W[..., 2, 0], W[..., 0, 1]
        omega = torch.stack([wx, wy, wz], dim=-1)
        return torch.cat([trS, a2, omega], dim=-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.enc(self.featurize(x))
        h = self.gate(h)

        for lin in self.layers:
            h = lin(h)
            h = self.gate(h)

        out = self.dec(h)  # (B, 1+5)
        tr0 = out[..., 0:1]
        a2  = out[..., 1:6]
        return _irreps_to_mandel(tr0, a2)

In [8]:
# --------------------
# TBNN (Pope 10-basis)
# --------------------

def _tbnn_invariants_and_basis(x9: torch.Tensor, eps: float = 1e-8) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute invariants (5) and Pope basis tensors (10) from gradU.

    Uses normalized S and W (robust default).

    Returns:
      inv:   (B,5)
      basis: (B,10,3,3)
      I:     (B,3,3)
    """
    U = x9.view(-1, 3, 3)
    S = 0.5 * (U + U.transpose(-1, -2))
    W = 0.5 * (U - U.transpose(-1, -2))

    nS = torch.sqrt((S*S).sum(dim=(-1,-2), keepdim=True)).clamp_min(eps)
    nW = torch.sqrt((W*W).sum(dim=(-1,-2), keepdim=True)).clamp_min(eps)
    Sh = S / nS
    Wh = W / nW

    I = torch.eye(3, device=x9.device, dtype=x9.dtype).view(1,3,3).expand(U.shape[0],3,3)

    S2 = Sh @ Sh
    S3 = S2 @ Sh
    W2 = Wh @ Wh

    # 5 invariants commonly used in TBNN literature
    trS2  = torch.einsum('bii->b', S2)
    trW2  = torch.einsum('bii->b', W2)
    trS3  = torch.einsum('bii->b', S3)
    trW2S = torch.einsum('bii->b', (W2 @ Sh))
    trW2S2= torch.einsum('bii->b', (W2 @ S2))
    inv = torch.stack([trS2, trW2, trS3, trW2S, trW2S2], dim=-1)

    # Pope's 10 basis tensors (symmetric)
    T1  = Sh
    T2  = Sh @ Wh - Wh @ Sh
    T3  = S2 - (trS2.view(-1,1,1)/3.0)*I
    T4  = W2 - (trW2.view(-1,1,1)/3.0)*I
    T5  = Wh @ S2 - S2 @ Wh
    T6  = W2 @ Sh + Sh @ W2 - (2.0/3.0)*torch.einsum('bii->b', Sh @ W2).view(-1,1,1)*I
    T7  = Wh @ Sh @ W2 - W2 @ Sh @ Wh
    T8  = Sh @ Wh @ S2 - S2 @ Wh @ Sh
    T9  = W2 @ S2 + S2 @ W2 - (2.0/3.0)*torch.einsum('bii->b', S2 @ W2).view(-1,1,1)*I
    T10 = Wh @ S2 @ W2 - W2 @ S2 @ Wh

    basis = torch.stack([T1,T2,T3,T4,T5,T6,T7,T8,T9,T10], dim=1)
    # numerically enforce symmetry (some terms are symmetric analytically; safe anyway)
    basis = 0.5 * (basis + basis.transpose(-1,-2))

    return inv, basis, I


class TBNNPointwise(nn.Module):
    """Tensor Basis Neural Network that predicts a symmetric tensor then maps to Mandel(6).

    We model:
      T_pred = alpha*I + sum_{i=1..10} c_i * T_i

    where T_i are Pope basis tensors computed from gradU.

    NOTE: This is a *full* 10-basis TBNN (not a reduced or randomized basis).
    """
    def __init__(self, hidden: int = 256, depth: int = 4, dropout: float = 0.0, layernorm: bool = True):
        super().__init__()
        in_dim = 5
        layers: List[nn.Module] = []
        d = in_dim
        for _ in range(depth):
            layers.append(nn.Linear(d, hidden))
            if layernorm:
                layers.append(nn.LayerNorm(hidden))
            layers.append(nn.SiLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            d = hidden
        layers.append(nn.Linear(d, 11))  # alpha + 10 coeffs
        self.coeff_net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        inv, basis, I = _tbnn_invariants_and_basis(x)
        coeff = self.coeff_net(inv)         # (B,11)
        alpha = coeff[:, :1]                # (B,1)
        c = coeff[:, 1:]                    # (B,10)
        T = alpha.view(-1,1,1) * I + torch.einsum('bi, bijk->bjk', c, basis)
        T = 0.5*(T + T.transpose(-1,-2))
        return mat_to_mandel(T)


In [9]:
# =============================================================================
# Metrics
# =============================================================================

@torch.no_grad()
def r2_mandel(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """Global R2 across all 6 Mandel components (same scalar R2 definition as your sandbox)."""
    # flatten all components
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    sse = torch.sum((yp - yt)**2).item()
    ybar = torch.mean(yt).item()
    sst = torch.sum((yt - ybar)**2).item()
    sst = max(sst, 1e-12)
    return 1.0 - sse/sst


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> Dict[str, float]:
    model.eval()
    sse = 0.0
    n = 0
    y_all = []
    yhat_all = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        yhat = model(xb)
        y_all.append(yb)
        yhat_all.append(yhat)
        sse += F.mse_loss(yhat, yb, reduction='sum').item()
        n += yb.numel()

    y_all = torch.cat(y_all, dim=0)
    yhat_all = torch.cat(yhat_all, dim=0)

    mse = sse / max(n, 1)
    r2 = r2_mandel(y_all, yhat_all)
    return {'MSE': mse, 'R2': r2}

In [10]:
# =============================================================================
# Training
# =============================================================================

@dataclass
class TrainConfig:
    batch_size: int = 256
    epochs: int = 400
    lr: float = 1e-3
    weight_decay: float = 1e-4

    # symmetry-mix: total loss = (1-gamma) * model_loss + gamma * scale * sym_pen
    gamma_sched: str = 'jump'   # constant|linear_warmup|cosine|exponential|jump
    gamma_max: float = 0.5
    gamma_delay: int = 0
    gamma_per_batch: bool = False

    # logging
    print_every: int = 50


def make_gamma_fn(cfg: TrainConfig, steps_per_epoch: int) -> Callable[[int], float]:
    name = cfg.gamma_sched.lower()
    if name == 'constant':
        return S.constant(cfg.gamma_max)
    if name == 'linear_warmup':
        # warmup over delay steps
        return S.linear_warmup(cfg.gamma_max, cfg.gamma_delay)
    if name == 'cosine':
        total = cfg.epochs * steps_per_epoch if cfg.gamma_per_batch else cfg.epochs
        return S.cosine(cfg.gamma_max, total)
    if name == 'exponential':
        # interpret delay as tau for convenience
        tau = max(cfg.gamma_delay, 1)
        return S.exponential(cfg.gamma_max, tau)
    if name == 'jump':
        return S.jump(cfg.gamma_max, cfg.gamma_delay)
    raise ValueError(f"Unknown gamma schedule: {cfg.gamma_sched}")

In [11]:
def train_baseline_mlp(
    split: DataSplit,
    *,
    cfg: TrainConfig,
    seed: int,
    device: torch.device,
    model_kwargs: Optional[Dict] = None,
) -> Tuple[nn.Module, Dict[str, float]]:
    """Train baseline MLP (no symmetry penalty)."""
    set_seed(seed)
    model_kwargs = model_kwargs or {}
    model = BaselineMLP(**model_kwargs).to(device)

    train_loader = DataLoader(TabularTensorDataset(split.X_train, split.Y_train),
                              batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    test_loader = DataLoader(TabularTensorDataset(split.X_test, split.Y_test),
                             batch_size=max(cfg.batch_size, 512), shuffle=False, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mse = nn.MSELoss()

    for ep in range(1, cfg.epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            yhat = model(xb)
            loss = mse(yhat, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        if (ep == 1) or (ep % cfg.print_every == 0) or (ep == cfg.epochs):
            tr = evaluate(model, train_loader, device)
            te = evaluate(model, test_loader, device)
            print(f"[baseline] ep {ep:04d}/{cfg.epochs} train R2={tr['R2']:.4f} test R2={te['R2']:.4f}")

    metrics = evaluate(model, test_loader, device)
    metrics_train = evaluate(model, train_loader, device)
    return model, metrics, metrics_train

In [12]:
def train_regularized_mlp(
    split: DataSplit,
    *,
    cfg: TrainConfig,
    seed: int,
    device: torch.device,
    model_kwargs: Optional[Dict] = None,
) -> Tuple[nn.Module, Dict[str, float]]:
    """Train baseline MLP with symdisc equivariance penalty.

    Uses forward_with_equivariance_penalty directly on the tabular batch x (no lifting/padding).
    """
    set_seed(seed)
    model_kwargs = model_kwargs or {}
    model = BaselineMLP(**model_kwargs).to(device)

    train_loader = DataLoader(TabularTensorDataset(split.X_train, split.Y_train),
                              batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    test_loader = DataLoader(TabularTensorDataset(split.X_test, split.Y_test),
                             batch_size=max(cfg.batch_size, 512), shuffle=False, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mse = nn.MSELoss()

    # vector fields
    X3 = build_single_node_generator_X3()
    Y3 = build_single_output_generator_Y3()

    steps_per_epoch = max(1, len(train_loader))
    gamma_fn = make_gamma_fn(cfg, steps_per_epoch)

    penalties_scaled = False
    scale = torch.tensor(1.0, device=device)
    global_step = 0

    for ep in range(1, cfg.epochs + 1):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            gamma = float(gamma_fn(global_step if cfg.gamma_per_batch else ep))

            if gamma > 0.0:
                yhat, sym_pen = forward_with_equivariance_penalty(
                    model=model,
                    X_in=[X3],
                    Y_out=[Y3],
                    x=xb,
                    loss=nn.MSELoss(),
                    sample_fields=None,
                    weights=[1.0],
                )
                loss_m = mse(yhat, yb)

                if not penalties_scaled:
                    denom = torch.clamp(sym_pen.detach(), min=1e-8)
                    if torch.isfinite(denom).all():
                        scale = (loss_m.detach() / denom).to(device)
                    penalties_scaled = True

                loss = (1.0 - gamma) * loss_m + gamma * (sym_pen * scale)
            else:
                yhat = model(xb)
                loss = mse(yhat, yb)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            global_step += 1

        if (ep == 1) or (ep % cfg.print_every == 0) or (ep == cfg.epochs):
            tr = evaluate(model, train_loader, device)
            te = evaluate(model, test_loader, device)
            print(f"[regularized] ep {ep:04d}/{cfg.epochs} train R2={tr['R2']:.4f} test R2={te['R2']:.4f} gamma={gamma:.3f} scale={float(scale):.3g}")

    metrics = evaluate(model, test_loader, device)
    metrics_train = evaluate(model, train_loader, device)
    return model, metrics, metrics_train

In [13]:
def train_e3nn(
    split: DataSplit,
    *,
    cfg: TrainConfig,
    seed: int,
    device: torch.device,
    model_kwargs: Optional[Dict] = None,
) -> Tuple[nn.Module, Dict[str, float]]:
    """Train the e3nn pointwise equivariant model."""
    if not _HAS_E3NN:
        raise ImportError(
            "e3nn is not available. Install e3nn to run this method. "
            f"Original import error: {_E3NN_IMPORT_ERR}"
        )

    set_seed(seed)
    model_kwargs = model_kwargs or {}
    model = E3NNPointwise(**model_kwargs).to(device)

    train_loader = DataLoader(TabularTensorDataset(split.X_train, split.Y_train),
                              batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    test_loader = DataLoader(TabularTensorDataset(split.X_test, split.Y_test),
                             batch_size=max(cfg.batch_size, 512), shuffle=False, drop_last=False)

    # slightly smaller lr tends to help stability with larger irreps
    lr = min(cfg.lr, 5e-4)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    mse = nn.MSELoss()

    for ep in range(1, cfg.epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            yhat = model(xb)
            loss = mse(yhat, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        if (ep == 1) or (ep % cfg.print_every == 0) or (ep == cfg.epochs):
            tr = evaluate(model, train_loader, device)
            te = evaluate(model, test_loader, device)
            print(f"[e3nn] ep {ep:04d}/{cfg.epochs} train R2={tr['R2']:.4f} test R2={te['R2']:.4f}")

    metrics = evaluate(model, test_loader, device)
    metrics_train = evaluate(model, train_loader, device)
    return model, metrics, metrics_train

In [14]:
def train_tbnn(
    split: DataSplit,
    *,
    cfg: TrainConfig,
    seed: int,
    device: torch.device,
    model_kwargs: Optional[Dict] = None,
) -> Tuple[nn.Module, Dict[str, float]]:
    """Train the full 10-basis TBNN."""
    set_seed(seed)
    model_kwargs = model_kwargs or {}
    model = TBNNPointwise(**model_kwargs).to(device)

    train_loader = DataLoader(TabularTensorDataset(split.X_train, split.Y_train),
                              batch_size=cfg.batch_size, shuffle=True, drop_last=False)
    test_loader = DataLoader(TabularTensorDataset(split.X_test, split.Y_test),
                             batch_size=max(cfg.batch_size, 512), shuffle=False, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    mse = nn.MSELoss()

    for ep in range(1, cfg.epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            yhat = model(xb)
            loss = mse(yhat, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        if (ep == 1) or (ep % cfg.print_every == 0) or (ep == cfg.epochs):
            tr = evaluate(model, train_loader, device)
            te = evaluate(model, test_loader, device)
            print(f"[tbnn] ep {ep:04d}/{cfg.epochs} train R2={tr['R2']:.4f} test R2={te['R2']:.4f}")

    metrics = evaluate(model, test_loader, device)
    metrics_train = evaluate(model, train_loader, device)
    return model, metrics, metrics_train

In [15]:
# =============================================================================
# Experiment helpers (manual-friendly)
# =============================================================================

def run_trials(
    *,
    method: str,
    split: DataSplit,
    cfg: TrainConfig,
    n_runs: int = 10,
    seed0: int = 1337,
    device: Optional[torch.device] = None,
    out_csv: str | Path = 'results.csv',
    model_kwargs: Optional[Dict] = None,
) -> List[Dict[str, float]]:
    """Run n_runs seeds for one method and write results to out_csv.

    method in {'baseline','regularized','e3nn','tbnn'}

    Returns list of dict rows.
    """
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    out_csv = Path(out_csv)
    model_kwargs = model_kwargs or {}

    train_fn_map = {
        'baseline': train_baseline_mlp,
        'regularized': train_regularized_mlp,
        'e3nn': train_e3nn,
        'tbnn': train_tbnn,
    }
    if method not in train_fn_map:
        raise ValueError(f"Unknown method: {method}")

    rows: List[Dict[str, float]] = []
    t0 = time.time()

    for r in range(n_runs):
        seed = seed0 + r
        print(f"\n=== {method} run {r+1}/{n_runs} (seed={seed}) ===")
        model, metrics, metrics_train = train_fn_map[method](
            split,
            cfg=cfg,
            seed=seed,
            device=device,
            model_kwargs=model_kwargs,
        )
        rows.append({
            'method': method,
            'run': r,
            'seed': seed,
            'test_R2': float(metrics['R2']),
            'test_MSE': float(metrics['MSE']),
            'train_R2': float(metrics_train['R2']),
            'train_MSE': float(metrics_train['MSE']),
        })

    # write CSV
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)

    print(f"\nWrote {len(rows)} rows to {out_csv} (elapsed {time.time()-t0:.1f}s)")
    return rows

In [16]:
# =============================================================================
# Example usage (manual):
# =============================================================================

# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# split = load_squareduct_pointwise('squareDuct_1100_mandel.csv', 'squareDuct_yz_grid.csv', dtype=torch.float32)
# cfg = TrainConfig(epochs=400, batch_size=256, lr=1e-3, weight_decay=1e-4, gamma_max=0.5, gamma_delay=0, print_every=50)
#
# # Run each method and write separate CSVs
# run_trials(method='baseline',    split=split, cfg=cfg, n_runs=10, out_csv='baseline_results.csv')
# run_trials(method='regularized', split=split, cfg=cfg, n_runs=10, out_csv='regularized_results.csv')
# run_trials(method='e3nn',        split=split, cfg=cfg, n_runs=10, out_csv='e3nn_results.csv',
#           model_kwargs={'hidden_spec':'128x0e + 64x1e + 64x2e + 32x3e', 'depth':4})
# run_trials(method='tbnn',        split=split, cfg=cfg, n_runs=10, out_csv='tbnn_results.csv')

In [17]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
split = load_squareduct_pointwise('data/squareDuct_1100_mandel.csv', 'data/squareDuct_yz_grid.csv', dtype=torch.float32)
cfg = TrainConfig(epochs=100, batch_size=256, lr=1e-3, weight_decay=1e-4, gamma_max=0.5, gamma_delay=0, print_every=10)

# Run each method and write separate CSVs
run_trials(method='baseline',    split=split, cfg=cfg, n_runs=10, out_csv='results/baseline_results.csv')
run_trials(method='regularized', split=split, cfg=cfg, n_runs=10, out_csv='results/regularized_results.csv')
run_trials(method='e3nn',        split=split, cfg=cfg, n_runs=10, out_csv='results/e3nn_results.csv',
          model_kwargs={'hidden_spec':'128x0e + 64x1e + 64x2e + 32x3e', 'depth':4})
           #model_kwargs={'hidden_spec':'256x0e + 128x1e + 128x2e + 64x3e', 'depth':6})
run_trials(method='tbnn',        split=split, cfg=cfg, n_runs=10, out_csv='results/tbnn_results.csv')


=== baseline run 1/10 (seed=1337) ===
[baseline] ep 0001/100 train R2=0.3963 test R2=0.2593
[baseline] ep 0010/100 train R2=0.9069 test R2=0.5124
[baseline] ep 0020/100 train R2=0.9439 test R2=0.7086
[baseline] ep 0030/100 train R2=0.9802 test R2=0.7010
[baseline] ep 0040/100 train R2=0.9916 test R2=0.6989
[baseline] ep 0050/100 train R2=0.9929 test R2=0.7084
[baseline] ep 0060/100 train R2=0.9924 test R2=0.7138
[baseline] ep 0070/100 train R2=0.9943 test R2=0.7133
[baseline] ep 0080/100 train R2=0.9954 test R2=0.7325
[baseline] ep 0090/100 train R2=0.9962 test R2=0.7294
[baseline] ep 0100/100 train R2=0.9970 test R2=0.7187

=== baseline run 2/10 (seed=1338) ===
[baseline] ep 0001/100 train R2=0.4558 test R2=0.2657
[baseline] ep 0010/100 train R2=0.9054 test R2=0.5044
[baseline] ep 0020/100 train R2=0.9360 test R2=0.7395
[baseline] ep 0030/100 train R2=0.9835 test R2=0.6911
[baseline] ep 0040/100 train R2=0.9926 test R2=0.7042
[baseline] ep 0050/100 train R2=0.9936 test R2=0.7280
[bas

[{'method': 'tbnn',
  'run': 0,
  'seed': 1337,
  'test_R2': 0.9702415887688896,
  'test_MSE': 0.9319289304591991,
  'train_R2': 0.9681743538779135,
  'train_MSE': 0.9990841945012411},
 {'method': 'tbnn',
  'run': 1,
  'seed': 1338,
  'test_R2': 0.9721324699648,
  'test_MSE': 0.8727131618393792,
  'train_R2': 0.9704936467582463,
  'train_MSE': 0.9262760281562805},
 {'method': 'tbnn',
  'run': 2,
  'seed': 1339,
  'test_R2': 0.9551664375173423,
  'test_MSE': 1.4040297711337055,
  'train_R2': 0.9536877698746252,
  'train_MSE': 1.453853170077006},
 {'method': 'tbnn',
  'run': 3,
  'seed': 1340,
  'test_R2': 0.9717773724263741,
  'test_MSE': 0.8838335319801613,
  'train_R2': 0.9699983524164031,
  'train_MSE': 0.9418245333212393},
 {'method': 'tbnn',
  'run': 4,
  'seed': 1341,
  'test_R2': 0.9717972569681751,
  'test_MSE': 0.8832108268031368,
  'train_R2': 0.969946007976522,
  'train_MSE': 0.9434676744319774},
 {'method': 'tbnn',
  'run': 5,
  'seed': 1342,
  'test_R2': 0.9721199482300822,

In [18]:
import pandas as pd

In [19]:
baseDF = pd.read_csv('results/baseline_results.csv')
regDF = pd.read_csv('results/regularized_results.csv')
e3nnDF = pd.read_csv('results/e3nn_results.csv')
tbnnDF = pd.read_csv('results/tbnn_results.csv')

In [20]:
baseDF.head()

,method,run,seed,test_R2,test_MSE,train_R2,train_MSE
0,baseline,0,1337,0.718659,8.810604,0.997025,0.093386
1,baseline,1,1338,0.769405,7.221415,0.995798,0.131900
2,baseline,2,1339,0.723084,8.672023,0.997094,0.091235
3,baseline,3,1340,0.733772,8.337317,0.993546,0.202605
4,baseline,4,1341,0.724410,8.630522,0.995926,0.127906


In [23]:
print(baseDF["test_R2"].mean())
print(baseDF["test_R2"].std())

0.7348739934970532
0.014729755956335808


In [24]:
print(regDF["test_R2"].mean())
print(regDF["test_R2"].std())

0.8865022728696728
0.004458488511119445


In [25]:
print(e3nnDF["test_R2"].mean())
print(e3nnDF["test_R2"].std())

0.7785226754968378
5.05605110970579e-05


In [26]:
print(tbnnDF["test_R2"].mean())
print(tbnnDF["test_R2"].std())

0.9700706917905857
0.0053144854562605275
